# 🏋️ Farhan Fitness Knowledge Worker

A RAG-based knowledge worker for **Farhan Fitness LLC** — Farhan's neighborhood strength and conditioning gym (mock knowledge base).

This notebook combines:
- **Ingestion**: Loading documents, creating chunks, and building embeddings
- **Answer**: RAG-based question answering with context retrieval
- **App**: Gradio interface for interactive chat


## 1. Setup and Imports


In [ ]:
# Standard library + UI + LangChain stack for RAG (load .env for OPENAI_API_KEY).
import os
from pathlib import Path

import gradio as gr
from dotenv import load_dotenv

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages
from langchain_core.documents import Document

load_dotenv(override=True)



## 2. Configuration


In [ ]:
# --- Model & retrieval ---
MODEL = "gpt-4.1-nano"  # Chat model for final answers
RETRIEVAL_K = 10  # How many chunks to pull into context (similarity search)

# Chroma persists vectors on disk under this folder (created next to the notebook).
DB_NAME = "vector_db"
# Root folder containing one subfolder per category (see KNOWLEDGE_SUBFOLDERS).
KNOWLEDGE_BASE = "knowledge-base"

# Explicit list: only these subfolders are ingested. Add a new folder name here + on disk to extend the KB.
KNOWLEDGE_SUBFOLDERS = [
    "company",
    "employees",
    "membership",
    "classes",
    "equipment",
    "policies",
]

# Embeddings model must match what you used when building Chroma (re-run ingestion if you change it).
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# Placeholder {context} is filled with retrieved chunk text before calling the LLM.
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Farhan Fitness LLC.
You are chatting with a user about Farhan Fitness LLC.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""



## 3. Ingestion Functions

These functions load markdown from each folder listed in `KNOWLEDGE_SUBFOLDERS` under `knowledge-base/`, split into chunks, and build the vector store. Add a new top-level folder and append its name to `KNOWLEDGE_SUBFOLDERS` to include it in ingestion.


In [ ]:
def fetch_documents():
    """Walk each category folder and load all Markdown files into LangChain Documents."""
    documents = []
    for sub in KNOWLEDGE_SUBFOLDERS:
        folder = Path(KNOWLEDGE_BASE) / sub
        if not folder.is_dir():
            print(f"Warning: missing knowledge folder: {folder}")
            continue
        # doc_type tags chunks for debugging / UI (which top-level category this came from).
        doc_type = sub
        loader = DirectoryLoader(
            str(folder), glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}
        )
        folder_docs = loader.load()
        for doc in folder_docs:
            doc.metadata["doc_type"] = doc_type
            documents.append(doc)
    return documents


def create_chunks(documents):
    """Smaller overlapping chunks retrieve more precisely than whole files."""
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
    chunks = text_splitter.split_documents(documents)
    return chunks


def create_vectorstore(chunks):
    """Embed chunks and store in Chroma; wipes previous collection if DB folder exists."""
    if os.path.exists(DB_NAME):
        Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

    vectorstore = Chroma.from_documents(
        documents=chunks, embedding=embeddings, persist_directory=DB_NAME
    )

    collection = vectorstore._collection
    count = collection.count()

    sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
    dimensions = len(sample_embedding)
    print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")
    return vectorstore



## 4. Run Ingestion

Execute this cell to load documents and create the vector database.


In [ ]:
# Run once after changing markdown or KNOWLEDGE_SUBFOLDERS (rebuilds the vector index).
documents = fetch_documents()
print(f"Loaded {len(documents)} documents")

chunks = create_chunks(documents)
print(f"Created {len(chunks)} chunks")

vectorstore = create_vectorstore(chunks)
print("Ingestion complete!")



## 5. RAG Answer Functions

These functions handle retrieving relevant context and generating answers using the LLM.


In [ ]:
# Reload persisted vectors (same embedding model as ingestion) and wire retriever + LLM.
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)


def fetch_context(question: str) -> list[Document]:
    """Similarity search: return the k most relevant chunks for this query string."""
    return retriever.invoke(question, k=RETRIEVAL_K)


def combined_question(question: str, history: list[dict] = []) -> str:
    """Concatenate prior user utterances so retrieval sees multi-turn context."""
    prior = "\n".join(m["content"] for m in history if m["role"] == "user")
    return prior + "\n" + question


def answer_question(question: str, history: list[dict] = []) -> tuple[str, list[Document]]:
    """RAG core: retrieve → stuff into system prompt → chat completion."""
    combined = combined_question(question, history)
    docs = fetch_context(combined)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    messages = [SystemMessage(content=system_prompt)]
    messages.extend(convert_to_messages(history))
    messages.append(HumanMessage(content=question))
    response = llm.invoke(messages)
    return response.content, docs



## 6. Test the RAG System

Test the question answering before launching the full UI.


In [ ]:
# Quick sanity check without the UI (requires prior cells: config, ingest, RAG).
test_question = "What is Farhan Fitness and when was it founded?"
answer, context_docs = answer_question(test_question)

print("Question:", test_question)
print("\nAnswer:", answer)
print(f"\nRetrieved {len(context_docs)} context documents")



## 7. Gradio Chat Interface

The interactive chat application for the Farhan Fitness Knowledge Worker.


In [ ]:
def format_context(context):
    """Turn retrieved Documents into HTML for the Gradio Markdown panel."""
    result = "<h2 style='color: #ff7800;'>Relevant Context</h2>\n\n"
    for doc in context:
        result += f"<span style='color: #ff7800;'>Source: {doc.metadata['source']}</span>\n\n"
        result += doc.page_content + "\n\n"
    return result


def chat(history):
    """Gradio message-mode: last message is the new user turn; prior entries are context for RAG."""
    last_message = history[-1]["content"]
    prior = history[:-1]
    answer, context = answer_question(last_message, prior)
    history.append({"role": "assistant", "content": answer})
    return history, format_context(context)


def put_message_in_chatbot(message, history):
    """Two-step flow: append user message, then `chat` generates assistant reply + sources."""
    return "", history + [{"role": "user", "content": message}]



## 8. Launch the Application

Run this cell to launch the Gradio chat interface.


In [ ]:
# Gradio Blocks: chatbot + retrieved context side-by-side; submit chains two handlers.
theme = gr.themes.Soft(font=["Inter", "system-ui", "sans-serif"])

with gr.Blocks(title="Farhan Fitness Knowledge Worker", theme=theme) as ui:
    gr.Markdown("# 🏋️ Farhan Fitness Knowledge Worker\nAsk me anything about Farhan Fitness!")

    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(
                label="💬 Conversation", height=600, type="messages", show_copy_button=True
            )
            message = gr.Textbox(
                label="Your Question",
                placeholder="Ask anything about Farhan Fitness...",
                show_label=False,
            )

        with gr.Column(scale=1):
            context_markdown = gr.Markdown(
                label="📚 Retrieved Context",
                value="*Retrieved context will appear here*",
                container=True,
                height=600,
            )

    # First append user message to history, then run RAG and refresh context panel.
    message.submit(
        put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]
    ).then(chat, inputs=chatbot, outputs=[chatbot, context_markdown])

ui.launch(inbrowser=True)

